# Synthetic SaaS Signup Experiment

**GitHub + ChatGPT/Codex + Google Colab starter**

This notebook generates **fictional visitor-level data**, performs a frequentist two-proportion A/B test, and renders publication-friendly charts. No real customer data is used.

Run cells in order using **Runtime → Run all**. To use another scenario, adjust the experiment parameters below.

[![View source on GitHub](https://img.shields.io/badge/GitHub-view%20source-24292f?logo=github)](https://github.com/javidjamae/chatgpt-github-colab-workflow)

## 1. Environment setup

In Google Colab, the next cell clones this repository at the selected branch (`main` by default). In GitHub Actions, or when executing in a local clone, it uses the existing checkout. You can pin to a release tag for class material.

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/javidjamae/chatgpt-github-colab-workflow.git"
REPO_REF = "main"  # Change to a release tag, branch, or commit if needed
if not Path("src/experiment.py").exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, "workflow_repo"], check=True)
    os.chdir("workflow_repo")
    sys.path.insert(0, str(Path.cwd()))
else:
    sys.path.insert(0, str(Path.cwd()))
print("Project directory:", Path.cwd())

## 2. Choose a reproducible scenario

The *relative lift* is the multiplicative change to each visitor's baseline signup probability. The effect is built into the synthetic data-generating process, but observed results still vary with random sampling.

In [ ]:
from src.experiment import make_data, summarize, compare
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SAMPLE_SIZE = 12000
SEED = 42
RELATIVE_LIFT = 0.20

df = make_data(n=SAMPLE_SIZE, seed=SEED, lift=RELATIVE_LIFT)
assert df.visitor_id.is_unique
assert len(df) == SAMPLE_SIZE
print(df.head(8).to_string(index=False))

## 3. Explore the synthetic sample

Compare device mix and signup rates by variant. All figures are illustrative and should not be interpreted as actual product performance.

In [ ]:
display(pd.crosstab(df["variant"], df["device"], normalize="index").round(3))
summary = summarize(df)
display(summary.rename(columns={"sum":"signups", "count":"visitors", "mean":"signup_rate"}).round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
summary["mean"].mul(100).plot.bar(ax=axes[0], color=["#2563eb", "#f97316"], rot=0)
axes[0].set_title("Signup conversion by variant")
axes[0].set_ylabel("Conversion rate (%)")
axes[0].set_xlabel("")
by_device = df.groupby(["device", "variant"])["converted"].mean().unstack().mul(100)
by_device[["Control","Treatment"]].plot.bar(ax=axes[1], color=["#2563eb", "#f97316"], rot=0)
axes[1].set_title("Signup conversion by device")
axes[1].set_ylabel("Conversion rate (%)")
axes[1].set_xlabel("")
plt.tight_layout()
plt.show()

## 4. Test the observed conversion difference

A pooled, two-sided z-test assesses the null hypothesis of equal conversion rates. The 95% confidence interval for the **absolute** difference uses an unpooled Wald approximation. These are teaching examples: inspect sample sizes and expected counts before using normal approximations in real experiments. This notebook reports statistical evidence, not business impact or causal certainty beyond its randomized simulation.

In [ ]:
result = compare(df)
print(f"Control: {result['control_rate']:.2%} (n={result['n_control']:,})")
print(f"Treatment: {result['treatment_rate']:.2%} (n={result['n_treatment']:,})")
print(f"Absolute difference: {result['absolute_lift']*100:+.2f} percentage points")
print(f"Relative lift: {result['relative_lift']:+.2%}")
print(f"Two-sided p-value: {result['p_value']:.4g}")
print(f"95% CI: [{result['ci95'][0]*100:+.2f}, {result['ci95'][1]*100:+.2f}] percentage points")
assert 0 <= result["p_value"] <= 1

## 5. Visualize uncertainty

The confidence interval reflects estimated uncertainty around the absolute percentage-point effect. It is **not** a 95% probability that the fixed parameter lies inside this particular realized interval.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.5))
point = 100*result["absolute_lift"]
lo, hi = [100*x for x in result["ci95"]]
ax.errorbar(point, 0, xerr=[[point-lo], [hi-point]], fmt="o", capsize=7, color="#7c3aed", markersize=9)
ax.axvline(0, color="#475569", linestyle="--", label="No difference")
ax.set_yticks([])
ax.set_xlabel("Treatment - Control (percentage points)")
ax.set_title("Observed signup lift with 95% confidence interval")
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

## 6. Explore sampling variability

Simulate multiple independent experiments with the same underlying data-generating assumptions. See how the estimated difference varies even when the true relative lift is fixed. The chart is descriptive, not a substitute for a power analysis.

In [ ]:
N_SIMULATIONS = 200
simulation = np.array([compare(make_data(n=SAMPLE_SIZE, seed=SEED+i+1, lift=RELATIVE_LIFT))["absolute_lift"] for i in range(N_SIMULATIONS)])
fig, ax = plt.subplots(figsize=(10,4))
ax.hist(simulation * 100, bins=24, color="#8b5cf6", edgecolor="white")
ax.axvline(100*result["absolute_lift"], color="#ea580c", linewidth=2, label="Notebook sample")
ax.set_title("Variation in estimated lift across independent simulated experiments")
ax.set_xlabel("Observed absolute lift (percentage points)")
ax.set_ylabel("Simulation count")
ax.legend()
plt.tight_layout()
plt.show()
print(f"Average observed absolute lift across simulations: {simulation.mean()*100:+.2f} pp")

## 7. Optional: interactive controls in Colab

Run this cell to explore new scenarios without changing the earlier cells. If Colab does not render the widgets, you can still edit the configuration variables above and rerun cells manually.

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display
    def preview(n=6000, lift=0.20, seed=42):
        sample = make_data(n=n, lift=lift, seed=seed)
        s = summarize(sample)
        r = compare(sample)
        fig, ax = plt.subplots(figsize=(7, 3))
        s["mean"].mul(100).plot.bar(ax=ax, color=["#2563eb","#f97316"], rot=0)
        ax.set_title(f"Simulated signup rates | p = {r['p_value']:.3g}")
        ax.set_ylabel("Conversion (%)")
        ax.set_xlabel("")
        plt.tight_layout()
        plt.show()
    display(widgets.interactive(preview, n=widgets.IntSlider(value=6000, min=1000, max=20000, step=1000), lift=widgets.FloatSlider(value=0.20, min=-0.3, max=0.6, step=0.05), seed=widgets.IntSlider(value=42, min=1, max=100, step=1)))
except ImportError:
    print("Optional interactive widgets unavailable; the notebook remains runnable.")

## What to do next

- Edit sample size, seed, and treatment lift, then rerun.
- Fork the repository and add new experiments on a feature branch.
- Use ChatGPT or Codex to implement changes, run GitHub Actions, and publish a tagged notebook release.

See the [repository README](https://github.com/javidjamae/chatgpt-github-colab-workflow#readme) for the full end-to-end setup, including app connections and permissions.